In [0]:
# COMMAND --- NOTEBOOK

dbutils.widgets.text("catalog", "dev", "Target catalog")
catalog = dbutils.widgets.get("catalog")

# ---------------------------------------------------------------------------
# 1. Expectations History View (Corrected to taklaproducts pipeline logs)
# ---------------------------------------------------------------------------
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.taklaproducts.dq_expectations_history AS
SELECT
  origin.pipeline_name,
  origin.flow_name AS table_name,
  timestamp,
  exp.col.name AS rule_name,
  exp.col.dataset,
  exp.col.passed_records,
  exp.col.failed_records,
  ROUND(exp.col.failed_records / NULLIF(exp.col.passed_records + exp.col.failed_records, 0) * 100, 2) AS failure_rate_pct
FROM {catalog}.taklaproducts.ingestion_event_log_raw
CROSS JOIN LATERAL EXPLODE(
  FROM_JSON(details:flow_progress.data_quality.expectations,
    'array<struct<name:string, dataset:string, passed_records:int, failed_records:int>>')
) AS exp
WHERE event_type = 'flow_progress'

UNION ALL

SELECT
  origin.pipeline_name,
  origin.flow_name AS table_name,
  timestamp,
  exp.col.name AS rule_name,
  exp.col.dataset,
  exp.col.passed_records,
  exp.col.failed_records,
  ROUND(exp.col.failed_records / NULLIF(exp.col.passed_records + exp.col.failed_records, 0) * 100, 2) AS failure_rate_pct
FROM {catalog}.taklaproducts.transformation_event_log_raw
CROSS JOIN LATERAL EXPLODE(
  FROM_JSON(details:flow_progress.data_quality.expectations,
    'array<struct<name:string, dataset:string, passed_records:int, failed_records:int>>')
) AS exp
WHERE event_type = 'flow_progress'
""")

print(f"Created {catalog}.taklaproducts.dq_expectations_history")


# ---------------------------------------------------------------------------
# 2. Quarantine Counts View (Aligned exactly to your active architecture tables)
# ---------------------------------------------------------------------------
# Dynamic helper to build queries without throwing catalog lookup exceptions if tables are missing
def build_quarantine_select(table_name, target_table):
    full_path = f"{catalog}.taklaproducts.{target_table}"
    if spark.catalog.tableExists(full_path):
        return f"SELECT '{table_name}' AS source, COUNT(*) AS quarantined_count, current_timestamp() AS checked_at FROM {full_path}"
    else:
        return f"SELECT '{table_name}' AS source, 0 AS quarantined_count, current_timestamp() AS checked_at"

quarantine_queries = [
    build_quarantine_select("bronze_customers", "bronze_customers_quarantined"),
    build_quarantine_select("bronze_products", "bronze_products_quarantined"),
    build_quarantine_select("bronze_clickstream", "bronze_clickstream_quarantined"),
    build_quarantine_select("silver_customers", "silver_customers_quarantined"),
    build_quarantine_select("silver_products", "silver_products_quarantined"),
    build_quarantine_select("silver_clickstream", "silver_clickstream_quarantined"),
]

spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.taklaproducts.dq_quarantine_counts AS
{"\nUNION ALL\n".join(quarantine_queries)}
""")

print(f"Created {catalog}.taklaproducts.dq_quarantine_counts")


# ---------------------------------------------------------------------------
# 3. Referential Integrity View (Clickstream telemetry joins checked against dimensions)
# ---------------------------------------------------------------------------
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.taklaproducts.dq_referential_integrity AS
SELECT 'clickstream_to_products' AS relationship, COUNT(*) AS orphaned_count, current_timestamp() AS checked_at
FROM {catalog}.taklaproducts.silver_clickstream ev
WHERE ev.product_id IS NOT NULL
  AND NOT EXISTS (
    SELECT 1 
    FROM {catalog}.taklaproducts.silver_products p 
    WHERE p.product_id = ev.product_id
  )

UNION ALL
SELECT 'clickstream_to_customers', COUNT(*), current_timestamp()
FROM {catalog}.taklaproducts.silver_clickstream ev
WHERE ev.customer_id IS NOT NULL
  AND NOT EXISTS (
    SELECT 1 
    FROM {catalog}.taklaproducts.silver_customers c 
    WHERE c.customer_id = ev.customer_id 
      AND c.__END_AT IS NULL
  )
""")

print(f"Created {catalog}.taklaproducts.dq_referential_integrity")
